# Importing Packages

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col, when, to_date, trim, round, coalesce
from pyspark.sql.functions import count, to_timestamp, regexp_replace, lag, sum, date_trunc,year, lit, countDistinct
from pyspark.sql.window import Window
from pyspark.sql.types import *
load_dotenv()

# Defining Variables

In [0]:
GOLD_SCHEMA_PATH=os.getenv('GOLD_SCHEMA_PATH')

# Loading Fact and Dim Tables

In [0]:
dim_customer=spark.read.table(f"""{GOLD_SCHEMA_PATH}.`dim_customer`""")
dim_product=spark.read.table(f"""{GOLD_SCHEMA_PATH}.`dim_product`""")
dim_event=spark.read.table(f"""{GOLD_SCHEMA_PATH}.`dim_event`""")
fact_subscription=spark.read.table(f"""{GOLD_SCHEMA_PATH}.`fact_subscription`""")
subscription_analytics=spark.read.table(f"""{GOLD_SCHEMA_PATH}.`subscription_analytics`""")

# Current Day

In [0]:
current_day='2024-06-01'

# KPI 1 : Customer Loss or Gain

**Optimization in Joining**

In [0]:
total_customer_loss_and_gain=subscription_analytics\
    .select("customer_id","close_status","start_date")\
    .filter(year(col("start_date")) == year(to_date(lit(current_day))))\
    .groupBy("close_status").agg(countDistinct("customer_id").alias("customer_count"))
    

In [0]:
gain_count = total_customer_loss_and_gain.filter(col("close_status")=="Won").collect()[0][1]
loss_count = total_customer_loss_and_gain.filter(col("close_status")=="Lost").collect()[0][1]
total_customer=(
    gain_count - loss_count
)
print("Total Customer Gain or Loss is ",total_customer)

In [0]:
total_customer_loss_and_gain.display()

# KPI 2 : Customer Loss Percentage

In [0]:
current_year=year(to_date(lit(current_day)))
prev_year=current_year-1
condition=(year(col("start_date")).isin(prev_year,current_year)) & (col("close_status")==lit("Lost"))

In [0]:
loss_count=subscription_analytics.filter(condition)\
    .groupBy(year(col("start_date")).alias("Years"))\
    .agg(countDistinct("customer_id").alias("customer_count"))\
    .orderBy(col("Years").desc())
loss_count.display()

In [0]:
current_year_loss=loss_count.collect()[0][1]
prev_year_loss=loss_count.collect()[1][1]

In [0]:
loss_percentage=(current_year_loss-prev_year_loss)*100.0/prev_year_loss
print(f"This year loss percentage : {loss_percentage:.2f}%")

# KPI 3 : Highest-value customers by recurring revenue

In [0]:
highest_recurring_revenue=subscription_analytics\
    .groupBy("customer_id","customer_name")\
    .agg(round(sum("mrr_in_gpb"), 2).alias("recurring_revenue"))\
    .orderBy(col("recurring_revenue").desc())
highest_recurring_revenue.display()

# KPI 4 : Customer Stopping Product Usage

In [0]:
customer_events=subscription_analytics.select("customer_id","customer_name","product_id","close_status","Month")\
    .groupBy("customer_id","customer_name","close_status","Month")\
        .agg(countDistinct("product_id").alias("customer_count"))\
        .orderBy(col("customer_count").desc()).filter(col("close_status").isin("Lost","Won"))

In [0]:
downselling_customers=customer_events.filter(col("close_status")=="Lost")
upselling_customers=customer_events.filter(col("close_status")=="Won")

In [0]:
downselling_customers.createOrReplaceTempView("downselling_customers")
upselling_customers.createOrReplaceTempView("upselling_customers")

In [0]:
%sql
with downselling_per_month as (
-- Created rolling sum for each customer who are downselling
select customer_id, customer_name, month, sum(customer_count) over(partition by customer_id, customer_name order by month)
as total_product_reduced_per_month ,
row_number() over(partition by customer_id, customer_name order by month desc) as rn from 
downselling_customers order by customer_id,month)
-- Querying the top 10 customers who are downselling the most
select customer_id,customer_name,total_product_reduced_per_month from downselling_per_month where rn=1 
order by total_product_reduced_per_month desc limit 10

# KPI 5 : Customer Expanding Product Usage

In [0]:
%sql
with upselling_per_month as (
-- Created rolling sum for each customer who are upselling
select customer_id, customer_name, month, sum(customer_count) over(partition by customer_id, customer_name order by month)
as total_product_upselled_per_month ,
row_number() over(partition by customer_id, customer_name order by month desc) as rn from 
upselling_customers order by customer_id,month)
-- Querying the top 10 customers who are upselling the most
select customer_id,customer_name,total_product_upselled_per_month from upselling_per_month where rn=1 
order by total_product_upselled_per_month desc limit 10

#  KPI 6 : Recurring revenue growth from upgrades vs losses in current FY

In [0]:
downselling_mrr=downselling_customers\
    .filter(year(col("Month")) == year(to_date(lit(current_day))))\
    .join(subscription_analytics,["customer_id","customer_name","Month","close_status"])\
    .select("Month","mrr_in_gpb")\
    .groupBy("Month")\
    .agg(round(sum("mrr_in_gpb"),2).alias("downselling_mrr"))\
    .orderBy(col("Month").desc())

In [0]:
upselling_mrr=upselling_customers\
    .filter(year(col("Month")) == year(to_date(lit(current_day))))\
    .join(subscription_analytics,["customer_id","customer_name","Month","close_status"])\
    .select("Month","mrr_in_gpb")\
    .groupBy("Month")\
    .agg(round(sum("mrr_in_gpb"),2).alias("upselling_mrr"))\
    .orderBy(col("Month").desc())

In [0]:
diff_in_upselling_and_downselling=upselling_mrr.join(downselling_mrr,"Month").withColumn("Difference Amount",col("upselling_mrr")-col("downselling_mrr"))
diff_in_upselling_and_downselling.orderBy(col("Month").desc()).display()

# KPI 7 : Recurring revenue from existing customer

In [0]:
subscription_analytics\
    .select("customer_id","start_date","mrr_in_gpb")\
    .createOrReplaceTempView("recurring_revenue_from_exising_customers")


In [0]:
existing_customers = spark.sql(f"""
SELECT DISTINCT customer_id
FROM recurring_revenue_from_exising_customers
WHERE start_date < date_format(concat(year('{current_day}'),'-','01','-','01'),'yyyy-MM-dd')
""")

In [0]:
start_mrr = spark.sql(f"""
SELECT
    customer_id,
    MAX_BY(mrr_in_gpb, start_date) AS start_mrr
FROM recurring_revenue_from_exising_customers
WHERE start_date < date_format(concat(year('{current_day}'),'-','01','-','01'),'yyyy-MM-dd')
GROUP BY customer_id
""")

In [0]:
current_mrr = spark.sql(f"""
SELECT
    customer_id,
    SUM(mrr_in_gpb) AS current_mrr
FROM recurring_revenue_from_exising_customers
WHERE start_date BETWEEN 
    date_format(concat(year('{current_day}'),'-','01','-','01'),'yyyy-MM-dd')
    AND '{current_day}'
GROUP BY customer_id
""")

In [0]:
retention_df = start_mrr.alias("s") \
    .join(current_mrr.alias("c"), "customer_id", "inner") \
    .selectExpr(
        "SUM(c.current_mrr) / SUM(s.start_mrr) AS retention_ratio"
    )

In [0]:
retention_df.show()

# KPI 8 : revenue year-to-date 

In [0]:
ytd_revenue = spark.sql(f"""
SELECT
    SUM(mrr_in_gpb) AS ytd_revenue
FROM recurring_revenue_from_exising_customers
WHERE start_date BETWEEN date_format(concat(year('{current_day}'),'-','01','-','01'),'yyyy-MM-dd') AND '{current_day}'
""")

In [0]:
last_year_ytd_revenue = spark.sql(f"""
SELECT
    SUM(mrr_in_gpb) AS last_year_ytd_revenue
FROM recurring_revenue_from_exising_customers
WHERE start_date BETWEEN date_format(concat(year(date_add('{current_day}', -365)), '-', '01', '-', '01'), 'yyyy-MM-dd')
    AND date_add(date_format(concat(year(date_add('{current_day}', -365)), '-', '01', '-', '01'), 'yyyy-MM-dd'), dayofyear('{current_day}') - 1)
""")

In [0]:
comparison_df = ytd_revenue.crossJoin(last_year_ytd_revenue) \
    .withColumn("ytd_difference", col("ytd_revenue") - col("last_year_ytd_revenue")) \
    .withColumn("ytd_growth_pct", (col("ytd_difference") / col("last_year_ytd_revenue")) * 100)

display(comparison_df)

# KPI 9 : customers or products contribute the most to total recurring revenue

In [0]:
top_customers_products = subscription_analytics \
    .groupBy("customer_id", "product_id") \
    .agg(round(sum("mrr_in_gpb"),2)\
    .alias("total_mrr"))\
    .orderBy(col("total_mrr").desc())\
    .limit(10)


display(top_customers_products)

# KPI 10 : Rolling 12 month recurring revenue

In [0]:
subscription_analytics.select("Month","mrr_in_gpb").createOrReplaceTempView("rolling_mrr")

In [0]:
%sql
select Month, sum(mrr_in_gpb) as monthly_mrr,
       sum(sum(mrr_in_gpb)) over(order by Month rows between 11 preceding and current row) as rolling_12_month_mrr
from rolling_mrr
group by Month
order by Month